# Remesher Demo 00 — Comfy3D setup + model cache

Run this notebook once per pod/machine before the workflow notebooks. It creates the shared `/workspace/input`, `/workspace/output`, and `/workspace/models` directories; optionally captures a Hugging Face read token for gated models; writes `config.json`; starts or connects to Comfy3D via `pull-comfy3d.sh`; downloads or verifies the Remesher model groups; and checks the CLI/ComfyUI connection.

After this passes, continue with notebook 01, 02, 03, or 04 without repeating model downloads.

Create a Hugging Face **read** token here before running gated model downloads:

[Create HF read token](https://huggingface.co/settings/tokens/new?tokenType=read)

In [ ]:
from getpass import getpass
import json, os, pathlib, subprocess
WORKSPACE = pathlib.Path('/workspace')
ROOT = pathlib.Path('/workspace/remesher')
INPUT = pathlib.Path('/workspace/input')
OUTPUT = pathlib.Path('/workspace/output')
MODELS = pathlib.Path('/workspace/models')
for p in [INPUT, OUTPUT, MODELS]: p.mkdir(parents=True, exist_ok=True)
print('ROOT', ROOT); print('INPUT', INPUT); print('OUTPUT', OUTPUT)

In [ ]:
print('Create a Hugging Face read token if needed: https://huggingface.co/settings/tokens/new?tokenType=read')
hf = getpass('HF token for gated Hugging Face models (leave blank to skip): ')
if hf:
    os.environ['HF_TOKEN'] = hf
    print('HF_TOKEN set for this notebook process only.')
else:
    print('No HF token set. Public models only; gated downloads may fail.')

In [ ]:
config = {'server_url': 'http://host.docker.internal:8188/'}
(ROOT / 'config.json').write_text(json.dumps(config, indent=2))
print((ROOT / 'config.json').read_text())

In [ ]:
subprocess.run(['bash', 'docker/demo-jupyter/scripts/pull-comfy3d.sh'], cwd=ROOT, check=True)

## Download the exact Remesher workflow models

This cell downloads/preflights the exact assets used by the Remesher examples:

- `qwenimage2512`: `qwen_image_2512_fp8_e4m3fn`, Qwen 2.5 VL text encoder, Qwen VAE, Qwen Lightning LoRA
- `qwenimageedit2511`: `qwen_image_edit_2511_fp8mixed`, Qwen Edit Lightning LoRA
- `trellis2`: `microsoft/TRELLIS.2-4B` plus `microsoft/TRELLIS-image-large` sparse-structure decoder
- `dinov3`: `facebook/dinov3-vitl16-pretrain-lvd1689m` cloned into `models/facebook/dinov3-vitl16-pretrain-lvd1689m/`
- `mia`: Make-It-Animatable `bw.pth`, `bw_normal.pth`, `joints.pth`, `joints_coarse.pth`, `pose.pth`

1. DINOv3 requires a Hugging Face read token. Create one here: [Create HF read token](https://huggingface.co/settings/tokens/new?tokenType=read).
2. Request access to the gated DINOv3 model [here](https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m)



In [ ]:
# Download required Qwen/Trellis2/DINOv3/MIA models inside the Comfy3D container.
# This is intentionally Remesher-specific; do not use the generic /app/utils/model_config.json here.
import os, subprocess, textwrap

# Make this cell robust when run out-of-order or after a kernel restart.
COMFY_CONTAINER = globals().get('COMFY_CONTAINER') or os.environ.get('COMFY_CONTAINER', 'remesher-comfy3d')
container = COMFY_CONTAINER
script_host = '/workspace/remesher/docker/demo-jupyter/scripts/download-remesher-models.py'
script_container = '/tmp/download-remesher-models.py'
groups = os.environ.get('REMESHER_MODEL_GROUPS', 'all')  # e.g. qwenimage2512,trellis2,dinov3,mia

subprocess.run(['docker', 'cp', script_host, f'{container}:{script_container}'], check=True)
# Pass the HF token through docker exec environment flags instead of embedding it in
# the shell command string. This avoids leaking the token in CalledProcessError reprs.
def clean_hf_token(value):
    if not value:
        return ''
    value = str(value).strip().strip('\"').strip("'")
    if not value:
        return ''
    if not value.startswith('hf_') or any(ord(ch) > 127 for ch in value) or any(ch.isspace() for ch in value):
        print('Ignoring invalid HF token. Paste only the raw token value starting with hf_.')
        return ''
    return value

hf_clean = clean_hf_token(hf)
cmd = f"python {script_container} --groups {groups}"
exec_cmd = ['docker', 'exec']
if hf_clean:
    exec_cmd += ['-e', f'HF_TOKEN={hf_clean}', '-e', f'HUGGINGFACE_TOKEN={hf_clean}']
else:
    # Prevent stale/invalid host or notebook env vars from being inherited by docker exec.
    exec_cmd += ['-e', 'HF_TOKEN=', '-e', 'HUGGINGFACE_TOKEN=']
exec_cmd += [container, 'bash', '-lc', cmd]
print('Running in', container, ':', cmd)
subprocess.run(exec_cmd, check=True)


In [ ]:
# Show available downloader configs and key model cache folders for debugging.
subprocess.run([
    'docker', 'exec', 'remesher-comfy3d', 'bash', '-lc',
    'find /app /workspace -maxdepth 5 -type f \( -name "model_config.json" -o -iname "*model*download*.py" \) | sort; du -sh /app/models /workspace/ComfyUI/models 2>/dev/null || true'
], cwd=ROOT, check=True)


In [ ]:
subprocess.run(['comfy-prompt-cli', '--help'], cwd=ROOT, check=True)
subprocess.run(['comfy-prompt-cli', 'health', '--config', 'config.json'], cwd=ROOT, check=True)